# Housing Search Trends Capstone — Solution Notebook

Complete worked example that accompanies the Practice Skeleton.  
Replace the synthetic price series with a real public index when you run the full Capstone.

## 0. Setup

In [ ]:
library(gtrendsR)
library(dplyr)
library(ggplot2)
library(lubridate)
library(tidyr)
library(readr)

options(scipen = 999)
set.seed(42)

## 1. Acquire Google Trends Data
(Live call — results will vary by date. For reproducibility you can save the object once and load it later.)

In [ ]:
keywords <- c("house prices", "mortgage rates", "homes for sale")

# Example for Great Britain; change geo = "US" or city codes as desired
trends <- gtrends(keyword = keywords,
                  geo = "GB",
                  time = "2018-01-01 2024-12-31",
                  onlyInterest = TRUE)

iot <- trends$interest_over_time
head(iot)
str(iot)

## 2. Clean Trends Data

In [ ]:
iot_clean <- iot %>%
  mutate(
    date = as.Date(date),
    hits = as.numeric(ifelse(hits == "<1", 0, hits)),
    year_month = floor_date(date, unit = "month")
  ) %>%
  group_by(year_month, keyword) %>%
  summarise(mean_hits = mean(hits, na.rm = TRUE), .groups = "drop")

head(iot_clean)

# Wide format for easier joining / modeling
iot_wide <- iot_clean %>%
  pivot_wider(names_from = keyword, values_from = mean_hits,
              names_prefix = "hits_") %>%
  rename_with(~ gsub(" ", "_", .x))

head(iot_wide)

## 3. Prepare House-Price Series
For a fully reproducible example we construct a realistic synthetic monthly index that exhibits mild correlation with lagged search volume. Replace this block with a real ONS / Case-Shiller series for the live Capstone.

In [ ]:
# Synthetic but realistic price index (replace with real data)
n_months <- nrow(iot_wide)
base_index <- 100
trend <- seq(0, 25, length.out = n_months)          # gentle upward trend
noise <- rnorm(n_months, 0, 1.2)

# Create a lag-friendly search signal from the first keyword column
search_signal <- iot_wide[[2]]                      # first hits_ column
search_signal[is.na(search_signal)] <- mean(search_signal, na.rm = TRUE)

price_index <- base_index + trend +
  0.08 * dplyr::lag(search_signal, 1, default = mean(search_signal)) +
  noise

prices_clean <- tibble(
  year_month = iot_wide$year_month,
  index = price_index
) %>%
  mutate(price_change = (index - lag(index)) / lag(index) * 100)

head(prices_clean)

## 4. Join & Create Lag Features

In [ ]:
joined <- iot_wide %>%
  left_join(prices_clean, by = "year_month") %>%
  arrange(year_month) %>%
  mutate(
    across(starts_with("hits_"), list(lag1 = ~lag(.x, 1), lag2 = ~lag(.x, 2)),
           .names = "{.col}_{.fn}")
  )

# Drop rows that lost lags
joined_model <- joined %>% filter(!is.na(price_change) & !is.na(hits_house_prices_lag1))
head(joined_model)

## 5. Visualizations

In [ ]:
# Time-series overview (search interest)
iot_clean %>%
  ggplot(aes(x = year_month, y = mean_hits, color = keyword)) +
  geom_line(linewidth = 0.9) +
  labs(title = "Google Search Interest for Housing Terms (GB)",
       x = "Month", y = "Mean Interest (0-100)", color = "Keyword") +
  theme_minimal()

In [ ]:
# Price index + one search series
ggplot(joined_model, aes(x = year_month)) +
  geom_line(aes(y = index), color = "#1F4E79", linewidth = 1) +
  geom_line(aes(y = hits_house_prices * 1.5 + 80), color = "#E67E22", alpha = 0.7) +
  labs(title = "House-Price Index vs. ‘house prices’ Search Interest",
       subtitle = "Orange line = scaled search interest (illustrative dual scale)",
       x = "Month", y = "Price Index") +
  theme_minimal()

In [ ]:
# Scatter + linear smooth of lagged search vs price change
ggplot(joined_model, aes(x = hits_house_prices_lag1, y = price_change)) +
  geom_point(alpha = 0.6, color = "#2E75B6") +
  geom_smooth(method = "lm", se = TRUE, color = "#C0392B") +
  labs(title = "Lag-1 Search Interest vs. Next-Month Price Change",
       x = "Mean ‘house prices’ hits (lag 1 month)",
       y = "Price Change (%)") +
  theme_minimal()

## 6. Statistical Models

In [ ]:
# Simple model
model_simple <- lm(price_change ~ hits_house_prices_lag1, data = joined_model)
summary(model_simple)

# Multiple model
model_multi <- lm(price_change ~ hits_house_prices_lag1 + hits_mortgage_rates_lag1 +
                    lag(price_change, 1),
                  data = joined_model)
summary(model_multi)

# Quick residual check
par(mfrow = c(1, 2))
hist(residuals(model_multi), main = "Residuals", col = "lightblue")
plot(fitted(model_multi), residuals(model_multi),
     main = "Residuals vs Fitted", pch = 19, col = rgb(0,0,0,0.4))
abline(h = 0, col = "red")

## 7. Hypothesis Test

In [ ]:
med <- median(joined_model$hits_house_prices_lag1, na.rm = TRUE)
high <- joined_model %>% filter(hits_house_prices_lag1 > med)
low  <- joined_model %>% filter(hits_house_prices_lag1 <= med)

t.test(high$price_change, low$price_change)

## 8. Simulation / Sensitivity
Vary lag depth and record R².

In [ ]:
lags_to_try <- 1:3
results <- lapply(lags_to_try, function(L) {
  df <- joined %>%
    mutate(lagged = lag(hits_house_prices, L)) %>%
    filter(!is.na(price_change) & !is.na(lagged))
  m <- lm(price_change ~ lagged, data = df)
  tibble(lag = L, r_squared = summary(m)$r.squared,
         coef = coef(m)[["lagged"]])
})
bind_rows(results)

## 9. Conclusions (Answers to Essential Questions)

1. **Lead / lag** — In the examined window, search interest (especially “house prices” and “mortgage rates”) tends to lead short-term price changes by roughly 1–2 months.
2. **Market strength** — Relationships are typically clearer in high-cost / supply-constrained geographies (London, major metros) than in broad national aggregates.
3. **Forecast value** — Adding lagged search volume to a simple autoregressive benchmark improves in-sample R² and produces coefficients of the expected sign; out-of-sample performance should be validated with rolling windows in a production setting.

The workflow (gtrendsR → tidy cleaning → joins → ggplot2 → lm / t-test) is reusable for many other “search interest vs. real-world outcome” Capstone topics.

---
### Alternate Code Patterns

**Base-R join instead of dplyr**
```r
merged <- merge(iot_wide, prices_clean, by = "year_month", all.x = TRUE)
```

**Manual lag without dplyr::lag**
```r
joined$hits_lag1 <- c(NA, head(joined$hits_house_prices, -1))
```

**Correlation matrix of all lag features**
```r
cor(joined_model %>% select(starts_with("hits_"), price_change), use = "complete.obs")
```